# Understanding Edges and Filters in CNNs

In this notebook, we will explore the fundamental building blocks of Convolutional Neural Networks (CNNs):
1.  **What is an edge?**
2.  **How do filters detect edges?**
3.  **Interactive 10x10 Example**


## 1. What is an edge?

In the context of computer vision and images, an **edge** is a sudden, rapid change in pixel intensity. 

- On one side of the edge, pixels might be dark (low values).
- On the other side, pixels might be bright (high values).

Let's visualize a simple vertical edge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Create a 6x6 image with a vertical edge
image_edge = np.array([
    [10, 10, 10, 200, 200, 200],
    [10, 10, 10, 200, 200, 200],
    [10, 10, 10, 200, 200, 200],
    [10, 10, 10, 200, 200, 200],
    [10, 10, 10, 200, 200, 200],
    [10, 10, 10, 200, 200, 200]
])

plt.figure(figsize=(4, 4))
sns.heatmap(image_edge, annot=True, fmt="d", cmap="gray", cbar=False)
plt.title("An Image with a Vertical Edge")
plt.axis('off')
plt.show()

You can see clear separation between the dark pixels (10) on the left and bright pixels (200) on the right. That boundary is the **edge**.

## 2. How do filters detect edges?

A **filter** (or kernel) is a small matrix (e.g., 3x3) that we slide over the image. To detect a specific pattern, the filter's weights are designed to have a high response (output value) when they align with that pattern in the image.

Mathematically, this is a **convolution** operation (technically cross-correlation in many DL frameworks, but the concept is the same): we multiply the filter values with the overlapping image pixels and sum them up.

Consider this "Vertical Edge Detector" filter:

```
[[-1, 0, 1],
 [-1, 0, 1],
 [-1, 0, 1]]
```

Notice how it has negative values on the left and positive values on the right? It's designed to respond strongly when it sees dark pixels on the left and bright pixels on the right.

<video src="conv_animation.mp4" autoPlay controls loop>

In [ ]:
from scipy.signal import convolve2d

# Define a vertical edge filter (Sobel-like)
filter_vertical = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

print("Vertical Edge Filter:\n", filter_vertical)

# Apply convolution
# mode='valid' means we only compute where the filter fits completely inside the image
output = convolve2d(image_edge, filter_vertical, mode='valid')

plt.figure(figsize=(6, 4))
sns.heatmap(output, annot=True, fmt="d", cmap="viridis", cbar=True)
plt.title("Result of Convolution")
plt.axis('off')
plt.show()

The high values (570) in the output correspond to the location of the vertical edge in the original image. The filter has "detected" the edge!

## 3. Interactive 10x10 Example

Now let's play with a 10x10 image. 

**Instructions:**
1.  Run the cell below.
2.  Modify the `image_10x10` array to create different shapes (squares, lines, etc.).
3.  Modify the `my_filter` to see how different filters affect the output (e.g., try a horizontal edge filter).
4.  Re-run the cell to see the result.

In [ ]:
# --- INTERACTIVE SECTION ---

# 1. Create a 10x10 Image
# Let's draw a bright square in the middle of a dark background
image_10x10 = np.zeros((10, 10))
image_10x10[3:7, 3:7] = 100  # A 4x4 square with intensity 100

# 2. Define your Filter
# Try changing these values!
# This is a Horizontal Edge Detector
my_filter = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1]
])

# 3. Convolve
output_10x10 = convolve2d(image_10x10, my_filter, mode='valid')

# 4. Visualize Side-by-Side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Input Image
sns.heatmap(image_10x10, annot=True, fmt=".0f", cmap="gray", cbar=False, ax=axes[0], annot_kws={"size": 8})
axes[0].set_title("Input Image (10x10)")
axes[0].axis('off')

# Filter
sns.heatmap(my_filter, annot=True, fmt=".0f", cmap="coolwarm", cbar=False, ax=axes[1], annot_kws={"size": 12}, square=True)
axes[1].set_title("Filter (3x3)")
axes[1].axis('off')

# Output Feature Map
sns.heatmap(output_10x10, annot=True, fmt=".0f", cmap="viridis", cbar=False, ax=axes[2], annot_kws={"size": 8})
axes[2].set_title("Output Feature Map")
axes[2].axis('off')

plt.tight_layout()
plt.show()

### Questions to explore:
1.  Why are there positive and negative values in the output?
    *   *Hint: Look at the top edge of the square vs the bottom edge. One goes Dark->Bright, the other Bright->Dark.*
2.  What happens if you change `my_filter` to be all ones? `np.ones((3,3))`? 
    *   *Hint: This computes a local average (blurring).*
3.  Can you make a filter that detects corners?

### 5. The Dimensionality Problem
Notice something interesting about the shapes? 

The input image is **10x10**.
The filter is **3x3**.

What is the size of the output?
Let's check code:

In [ ]:
print(f"Input Shape: {image_10x10.shape}")
print(f"Output Shape: {output_10x10.shape}")

We lost 2 pixels in height and 2 pixels in width! The output is **8x8**.

This happens because the convolution operation (in 'valid' mode) only computes values where the filter fully overlaps with the image. It can't go off the edges.

### 6. Solution: Padding
To keep the output size the same as the input size (10x10), we can use **Padding**. 
We add a border of zeros around the image so the filter has space to center itself on the edge pixels.

Let's try convolution with `mode='same'` (which applies padding automatically).

In [ ]:
# 1. Convolve with 'same' mode (Padding)
output_padded = convolve2d(image_10x10, my_filter, mode='same')

print(f"Padded Output Shape: {output_padded.shape}")

# 2. Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Input Image
sns.heatmap(image_10x10, annot=True, fmt=".0f", cmap="gray", cbar=False, ax=axes[0])
axes[0].set_title("Input Image (10x10)")
axes[0].axis('off')

# Output Feature Map (Padded)
sns.heatmap(output_padded, annot=True, fmt=".0f", cmap="viridis", cbar=False, ax=axes[1])
axes[1].set_title("Output Feature Map (Same Size!)")
axes[1].axis('off')

plt.tight_layout()
plt.show()